# 1. Logistische Regression

---
## 1.1 Motivation

Wo sind die Grenzen der linearen/multiplen Regression?

* Nicht-lineare Zusammenhänge nur schwer abbildbar.
* Viele Kategorien oder Ausreißer → Modell instabil.
* Wenn Y keine Zahl ist, sondern „Klasse“ → Ansatz passt nicht mehr.
* Beispiel Titanic Datensatz: „Hat ein Passagier überlebt?“ = ja/nein.

> **Folgerung:** Wir brauchen etwas, das nicht nur *vorhersagt*, sondern *entscheidet*!

---
### Merke

> **Regression:** Schätzung einer Zahl.

> **Klassifikation:** Zuordnung zu einer Kategorie.

Beide sind Formen von **Supervised Learning** – aber mit unterschiedlichem Ziel.

> **lineare Regression:** *Abhängige Variable* ist eine *metrische Variable* (z.B. Gehalt, Verbrauch)

> **logistische Regression:** *Abhängige Variable* ist eine *dichtome Variable* (z.B. krank / nicht krank, guter Kreditnehmer / schlechter Kreditnehmer) 


---
### Notwendige Imports

Links zu den Dokumentationen (siehe unten)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.datasets import fetch_openml

## 1.2 Titanic Datensatz - Exploratory Data Analysis (EDA)

Klassischer Machine Learning Datensatz. Built-In Datensatz aus *seaborn* Python Library.

In [ ]:
df = sns.load_dataset("titanic")

## 1.3 Weitere interessante Beispiel-Datensätze

Sie bieten eine Übersicht über verschiedene Features und Datentypen, die in Datensätzen enthalten sein können. Die folgenden 3 Datensätze sind alle aus *seaborn* Python Library und können als Einzeiler importiert werden. Siehe hierzu auch [All Seaborn Built-in Datasets](https://www.kaggle.com/datasets/abdoomoh/all-seaborn-built-in-datasets/data).

> Welche der folgenden Datensätze bieten wohl gute Beispiele für logistische Regression?

* Health Expenditure: Health expenditure statistics across countries.

In [ ]:
df_hexp = sns.load_dataset("healthexp")

* Penguins: Data on penguin species and their features.

In [ ]:
df_pen = sns.load_dataset("penguins")

* Diamonds: Data on diamond properties including price, cut, and clarity.

In [ ]:
df_dia = sns.load_dataset("diamonds")

* Heart disease: Clinical and lifestyle attributes for individuals, with a binary target indicating the presence of heart disease.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

df_heart = pd.read_csv(url, names=columns)
# (0 = gesund, 1 = herzkrank)
df_heart['target'] = (df_heart['target'] > 0).astype(int)

* German Credit: Attributes for individualas that describe them as good or bad credit risks.

In [ ]:
credit_data = fetch_openml(name='credit-g', version=1, as_frame=True)

df_gcredit = credit_data.frame
# (0 = schlecht, 1 = gut)
df_gcredit['target'] = (df_gcredit['class'] == 'good').astype(int)

## 1.4 German Credit Datensatz

Ein Blick in die Daten. (Vorschlag: *Data Wrangler* Extension von Microsoft in VS Code)

In [ ]:
df_gcredit.head(20)

| Attribut | Typ | Name / Beschreibung | Ausprägungen / Kodierung |
| :--- | :--- | :--- | :--- |
| **Attribute 1** | qualitative | Status of existing checking account | **A11**: < 0 DM<br>**A12**: 0 <= ... < 200 DM<br>**A13**: >= 200 DM / salary assignments for at least 1 year<br>**A14**: no checking account |
| **Attribute 2** | numerical | Duration in month | *Numerischer Wert (Monate)* |
| **Attribute 3** | qualitative | Credit history | **A30**: no credits taken / all credits paid back duly<br>**A31**: all credits at this bank paid back duly<br>**A32**: existing credits paid back duly till now<br>**A33**: delay in paying off in the past<br>**A34**: critical account / other credits existing (not at this bank) |
| **Attribute 4** | qualitative | Purpose | **A40**: car (new)<br>**A41**: car (used)<br>**A42**: furniture/equipment<br>**A43**: radio/television<br>**A44**: domestic appliances<br>**A45**: repairs<br>**A46**: education<br>**A47**: vacation (does not exist?)<br>**A48**: retraining<br>**A49**: business<br>**A410**: others |
| **Attribute 5** | numerical | Credit amount | *Numerischer Wert (Kredithöhe)* |
| **Attribute 6** | qualitative | Savings account/bonds | **A61**: < 100 DM<br>**A62**: 100 <= ... < 500 DM<br>**A63**: 500 <= ... < 1000 DM<br>**A64**: >= 1000 DM<br>**A65**: unknown / no savings account |
| **Attribute 7** | qualitative | Present employment since | **A71**: unemployed<br>**A72**: < 1 year<br>**A73**: 1 <= ... < 4 years<br>**A74**: 4 <= ... < 7 years<br>**A75**: >= 7 years |
| **Attribute 8** | numerical | Installment rate | *Numerischer Wert (% des verfügbaren Einkommens)* |
| **Attribute 9** | qualitative | Personal status and sex | **A91**: male : divorced/separated<br>**A92**: female : divorced/separated/married<br>**A93**: male : single<br>**A94**: male : married/widowed<br>**A95**: female : single |
| **Attribute 10** | qualitative | Other debtors / guarantors | **A101**: none<br>**A102**: co-applicant<br>**A103**: guarantor |
| **Attribute 11** | numerical | Present residence since | *Numerischer Wert (Jahre am Wohnort)* |
| **Attribute 12** | qualitative | Property | **A121**: real estate<br>**A122**: building society savings agreement / life insurance<br>**A123**: car or other (not in attribute 6)<br>**A124**: unknown / no property |
| **Attribute 13** | numerical | Age in years | *Numerischer Wert (Alter)* |
| **Attribute 14** | qualitative | Other installment plans | **A141**: bank<br>**A142**: stores<br>**A143**: none |
| **Attribute 15** | qualitative | Housing | **A151**: rent<br>**A152**: own<br>**A153**: for free |
| **Attribute 16** | numerical | Number of existing credits | *Numerischer Wert (Anzahl Kredite bei dieser Bank)* |
| **Attribute 17** | qualitative | Job | **A171**: unemployed / unskilled - non-resident<br>**A172**: unskilled - resident<br>**A173**: skilled employee / official<br>**A174**: management / self-employed / highly qualified employee / officer |
| **Attribute 18** | numerical | Maintenance liabilities | *Numerischer Wert (Anzahl unterhaltspflichtiger Personen)* |
| **Attribute 19** | qualitative | Telephone | **A191**: none<br>**A192**: yes, registered under the customer's name |
| **Attribute 20** | qualitative | Foreign worker | **A201**: yes<br>**A202**: no |

In [ ]:
df_gcredit.shape

In [ ]:
df_gcredit.info()

In [ ]:
df_gcredit.dtypes

In [ ]:
df_gcredit.isnull().sum()

In [ ]:
df_gcredit.describe

In [ ]:
sns.countplot(data=df_gcredit, x="class")
plt.title("Survival Count")

In [ ]:
numeric_cols = ["duration", "credit_amount", "residence_since", "age", "existing_credits", "num_dependents"]

df_gcredit[numeric_cols].hist(bins=30, figsize=(12, 8))
plt.suptitle("Distribution of Numeric Features")
plt.tight_layout()
plt.show()

In [ ]:
for col in numeric_cols:
    sns.boxplot(x=df_gcredit[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

In [ ]:
categorical_cols = ["checking_status", "credit_history", "purpose", "personal_status", "other_parties", "property_magnitude", "other_payment_plans", "housing", "job", "own_telephone" ,"foreign_worker"]

for col in categorical_cols:
    sns.countplot(data=df_gcredit, x=col)
    plt.title(f"Countplot of {col}")
    plt.show()

In [ ]:
df_gcredit.hist(bins=30, figsize=(12, 8))
plt.tight_layout()
plt.show()

In [ ]:
for col in numeric_cols:
    sns.histplot(data=df_gcredit, x=col, hue="target", element="step", stat="density", common_norm=False)
    plt.title(f"{col} by Credit Risk")
    plt.show()

In [ ]:
sns.heatmap(df_gcredit.corr(numeric_only=True), annot=True, cmap="coolwarm")

In [ ]:
sns.pairplot(df_gcredit)

In [ ]:
sns.pairplot(df_gcredit, hue="target")

In [ ]:
for col in categorical_cols:
    sns.countplot(data=df_gcredit, x=col, hue="target")
    plt.title(f"{col} vs. Credit risk")
    plt.show()

## 1.5 Modelle
---

### 1.5.1 Lineare Regression

In [ ]:
target = "target"
features = ["duration", "age", "credit_amount"]

X = df_gcredit[features]
y = df_gcredit[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# X_train = X_train.dropna()
# y_train = y_train.loc[X_train.index]

# X_test = X_test.dropna()
# y_test = y_test.loc[X_test.index]

In [ ]:
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)

y_pred = lin_model.predict(X_test)

print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

# Plot predicted vs. actual
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Linear Regression Predictions")
plt.savefig("results/linear_regression_plot.png")
plt.show()

In [ ]:
df_gcredit.isnull().sum()

In [ ]:
import missingno as msno
msno.matrix(df_gcredit)

In [ ]:
df_gcredit.shape

### 1.5.2 Logistische Regression

In [ ]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

### Wichtige Begriffe / Metriken

(vgl. ab Folie 27 *Warum neue Metriken*)
* Confusion Matrix (siehe oben)
    - TP (true positive): **Guten** Kreditnehmer **korrekt** erkannt (139)
    - TN (true negative): **Schlechten** Kreditnehmer **korrekt** erkannt (8)
    - FP (false positive): **Schlechten** Kreditnehmer **nicht** erkannt (51)
    - FN (false negative): **Guten** Kreditnehmer **nicht** erkannt (2)

* Accuracy
    $= \frac{TP + TN}{TP + TN + FP + FN}$

* Precision
    $= \frac{TP}{TP + FP}$

* Recall 
    $= \frac{TP}{TP + FN}$

* F1
    $= 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2 \cdot TP}{2 \cdot TP + FP + FN}$

* Log Odds / Odds Ratio
    $$ \log\left(\frac{P(Y = 1)}{1 - P(Y = 1)}\right) = \beta_0 + \beta_1 X_1 + \dots + \beta_p X_p$$
    - Jeder $\beta_i$ beschreibt den Einfluss eines Merkmals auf die log-Odds
    - Wenn $\beta_i$ > 0: Wahrscheinlichkeit für $Y = 1$ steigt mit $Xi$
    - $e^{\beta_i}$ = Odds Ratio → Faktor, um den sich die Chancen ändern.


In [ ]:
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-Score:", f1_score(y_test, y_pred))


### Literatur

* James et al. 2023 *An Introduction to Statistical Learning : with Applications in Python* Springer, Kap. 4.3 *Logistic Regression*, Link: [An Introduction to Statistical Learning](https://www.statlearning.com/)

* Benny Botsch 2023 *Maschinelles Lernen - Grundlagen und Anwendungen* Springer, Kap. 5.1.2 *Logistische Regression*, Link: [Maschinelles Lernen - Grundlagen und Anwendungen](https://link.springer.com/book/10.1007/978-3-662-67277-8)

---
### Nützliche Links 
* [Scikit-Learn documentation](https://scikit-learn.org)
* [seaborn: statistical data visualization](https://seaborn.pydata.org/)
* [pandas documentation](https://pandas.pydata.org/docs/)
* [NumPy documentation](https://numpy.org/devdocs/)
* [Matplotlib 3.5.3 documentation](https://matplotlib.org/3.5.3/index.html)
* Video von DATAtab zu logistischer Regression (Youtube)